In [ ]:
import json
from unsloth import FastLanguageModel
# import torch

---

In [ ]:
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-bnb-4bit/snapshots/70d37f62f4f475a5a42dbbd5b7aad38f420e0960" # llama 3.1 8b bnb 4bit
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Llama-3.1-8B/snapshots/3f0d51f8e5640f98f1a96ea9044a0e55c0a83814" # llama 3.1 8b
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Meta-Llama-3.1-8B-Instruct-bnb-4bit/snapshots/f15c379fb32bb402fa06a7ae9aecb1febf4b79ec"
# local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Llama-3.1-8B-unsloth-bnb-4bit/snapshots/0a43e2563fa80ebc5eb093baa843c9cb65ab50d5"
local_model_path = "/home/fluffymelon/.cache/huggingface/hub/models--unsloth--Llama-3.1-8B-Instruct-unsloth-bnb-4bit/snapshots/3f09f44543858745a6af68161eded01dbcc536ad"

In [ ]:
# === Load model ===
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = local_model_path,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True
)
FastLanguageModel.for_inference(model)

In [ ]:
# === Create dialog chunks
def create_chunks(messages, window=6):
    questions = []
    dialog_contexts = []
    for i, message in enumerate(messages):
        if message['from'] == 'Соня' and '?' in message['text']:
            questions.append(message['text'])

            local_context = []
            for j in range(max(0, i - window), min(len(messages), i + window)):
                local_context.append(f'{messages[j]['from']}: {messages[j]['text']}')
            dialog_contexts.append('\n'.join(local_context))

    return questions, dialog_contexts

In [ ]:
# === Create dialog chunks
def create_chunks(messages, window=6):
    questions = []
    lesha_answers = []
    for i, message in enumerate(messages):
        if message['from'] == 'Соня' and '?' in message['text']:
            questions.append(message['text'])

            counter = 0
            local_context = []
            j = 0
            while counter <= window and i + j < len(messages):
                if messages[i + j]['from'] == 'Леша':
                    local_context.append(f'{counter + 1}.{messages[i + j]['text']}')
                    counter += 1
                j += 1
            lesha_answers.append('\n'.join(local_context))

    return questions, lesha_answers

In [ ]:
def generate_qa(question, dialog_context):
#     prompt = f"""You are a helpful assistant. Given a conversation, determine whether the sentence is a question or not.
# Dialog:
# {dialog_text}

# Output format:
# [
#   {{
#     "question": "...",
#     "answer": "..."
#   }}
# ]
# """

#     prompt = f"""Ты мой персональный ассистент. Тебе дан диалог между Лешей и Соней. Определи по контексту, является ли предложение, сказанное Соней, вопросом или нет. Если это вопрос, то найди ответ Леши на этот вопрос. Если Ответа нет, то верни "". Если предложение не явяется вопросом, то верни "".

# Диалог:
# {dialog_context}

# Предложение Сони:
# {question}

# Output format:
# [
#   {{
#     "question": "Здесь должно быть предложение Сони"
#     "answer": "Здесь должен быть ответ Леши"
#   }}
# ]
# """

    prompt = f"""You are a helpful assistant. Given a conversation between Lesha and Sonya, determine whether a message from Sonya is a question. If it is, find the response from Lesha that answers it. If there's no answer, return "".

Dialogue:
{dialog_context}

Message to analyze:
{question}

Output format:
[
  {{
    "question": "..."
    "answer": "..."
  }}
]

Your answer:"""

    print(prompt)

    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **input_ids,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded
    # crude way to extract JSON from output
    # start = decoded.find("[")
    # end = decoded.rfind("]") + 1
    # try:
    #     return json.loads(decoded[start:end])
    # except:
    #     return []

In [ ]:
messages

In [ ]:
with open("../data/processed/messages_2025-05-03.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

# questions, dialog_contexts = create_chunks(messages, window=10)
questions, lesha_answers = create_chunks(messages, window=10)

# for question, dialog_context in zip(questions, dialog_contexts):
#     qa_pairs = generate_qa(question, dialog_context)
#     # for qa in qa_pairs:
#     #     if "question" in qa and "answer" in qa:
#     #         all_qa.append(qa)

#     print(qa_pairs)
#     break

# all_qa = []
# for chunk in tqdm(chunks):
#     dialog = format_dialog(chunk)
#     qa_pairs = generate_qa(dialog)
#     for qa in qa_pairs:
#         if "question" in qa and "answer" in qa:
#             all_qa.append(qa)

# # Write to JSONL
# with open(output_path, "w", encoding="utf-8") as f:
#     for item in all_qa:
#         f.write(json.dumps(item, ensure_ascii=False) + "\n")

# print(f"Saved {len(all_qa)} QA pairs to {output_path}")

In [ ]:
# Ensure the EOS token ID is available
if hasattr(tokenizer, 'eos_token_id') and tokenizer.eos_token_id is not None:
    eos_id = tokenizer.eos_token_id
else:
    # Fallback or error if EOS token is not found
    eos_id = model.config.eos_token_id
    if eos_id is None:
        # Attempting a common Llama EOS token ID if absolutely necessary,
        # but ideally it should be found in the tokenizer/config.
        # You might need to verify this ID for your specific Llama3 variant.
        # For Llama 3, 128001 is often an EOS token.
        # Use with caution:
        # eos_id = 128001
        # print("Warning: Using fallback EOS token ID 128001")
            raise ValueError("Could not determine the EOS token ID from the tokenizer or model config.")


input_ids = tokenizer("You are my personal assistant.\n\n### Question: How are you doing?\n\n### Answer:", return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(
        **input_ids,
        max_new_tokens=32,
        # eos_token_id=eos_id,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        top_k=20,
        repetition_penalty=1.2
    )
decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded)

In [ ]:
decoded

In [ ]:
output.size()

In [ ]:
i = 100
question, lesha_answer = questions[i], lesha_answers[i]

# prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

# You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>
# Given a conversation between Lesha and Sonya, determine whether a message from Sonya is a question. If it is, find the response from Lesha that answers it. If there's no answer, return ""

# Dialogue:
# {dialog_context}

# Message to analyze:
# {question}

# <|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


# prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

# Ты мой персональный ассистент.<|eot_id|><|start_header_id|>user<|end_header_id|>
# Тебе дан диалог между Лешей и Соней. Определи по контексту, является ли предложение, сказанное Соней, вопросом или нет. Если это вопрос, то найди ответ Леши на этот вопрос. Если Ответа нет, то верни "". Если предложение не явяется вопросом, то верни "".

# Диалог:
# {dialog_context}

# Предложение Сони:
# {question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>Ответ Леши:"""


prompt = f"""<|start_header_id|>system<|end_header_id|>

Ты мой персональный помощник.<|eot_id|><|start_header_id|>user<|end_header_id|>
Выбери самые правдоподобные ответы Леши на вопрос Сони.

Вопрос Сони:
{question}

Варианты ответов Леши:
{lesha_answer}

Будь краток. В качестве ответа напиши только номер этого ответа. Если их несколько, то напиши все номера через пробел.|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


# prompt = f"""Ты мой персональный помощник. Выбери самый правдоподобный ответ Леши на вопрос Сони.

# Вопрос Сони:
# {question}

# Варианты ответов Леши:
# {lesha_answer}

# Твой ответ:"""


# prompt = f"""Ты мой персональный ассистент. Тебе дан диалог между Лешей и Соней. Определи по контексту, является ли предложение, сказанное Соней, вопросом или нет. Если это вопрос, то найди ответ Леши на этот вопрос. Если Ответа нет, то верни "". Если предложение не явяется вопросом, то верни "".

# Диалог:
# {dialog_context}

# Предложение Сони:
# {question}

# Output format:
# [
#   {{
#     "question": "Здесь должно быть предложение Сони"
#     "answer": "Здесь должен быть ответ Леши"
#   }}
# ]
# """

# Dialogue:
# {dialog_context}

# Message to analyze:
# {question}

# Your response:"""

# # Output format:
# #   {{
# #     "question": "..."
# #     "answer": "..."
# #   }}
# # print(prompt)

input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(
    **input_ids,
    # top_p=0.9,
    # top_k=20,
    # temperature=0.5,
    # repetition_penalty=1.2,
    max_new_tokens=64,
    # do_sample=True
)
decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print(decoded)

In [ ]:
# Define prompt (keep it short to test first)
prompt = """Ты мой персональный ассистент. Вопрос: "Кошка — это животное?" Твой ответ:"""
# prompt = """
# <|begin_of_text|><|start_header_id|>system<|end_header_id|>

# You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

# What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    # max_new_tokens=50,
    # temperature=0.5,
    # repetition_penalty=1.2,
    # top_p=0.9,
    # top_p=0.95,
    # top_k=20,
    # temperature=0.5,
    repetition_penalty=1.2,
    max_new_tokens=64,
    # do_sample=True,
    # return_full_text=False,
    # eos_token_id=tokenizer.eos_token_id,
)

# Decode output
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

In [ ]:
# Format the chat properly using LLaMA 3 format
def format_prompt(system, user):
    return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

system_message = "Ты ассистент, который отвечает на вопросы."
user_message = 'Простой вопрос: "Кошка — это животное?" Ответ:'

formatted_prompt = format_prompt(system_message, user_message)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        # temperature=0.7,
        # repetition_penalty=1.2,
        # top_p=0.9,
        # do_sample=True,
        # eos_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
# Чёткий prompt
def format_prompt(system, user):
    return f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{system}<|eot_id|><|start_header_id|>user<|end_header_id|>\n{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"

system_msg = "Ты ассистент, который отвечает на вопросы. Отвечай точно, кратко и по существу."
user_msg = 'Простой вопрос: "Кошка — это животное?" Ответ:'

prompt = format_prompt(system_msg, user_msg)

# Токенизация
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Генерация
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,  # отключает "творчество"
        temperature=0.0,  # максимально детерминированный
        eos_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)